# 🔥 PyTorch Deep Learning Curriculum
## Research-Grade Mastery: Autograd, Lightning, Hugging Face, TorchScript & ONNX

> **Framework:** PyTorch 2.x  
> **Level:** Beginner → Advanced → Research → Production  
> **Estimated Time:** 70–90 hours  

---

## 📋 Table of Contents

1. [PyTorch Fundamentals & Autograd](#1-fundamentals)
2. [nn.Module — Building Blocks](#2-nn-module)
3. [Custom Training Loop — from scratch](#3-training-loop)
4. [DataLoader & Custom Datasets](#4-dataloaders)
5. [Computer Vision — CNNs & ResNet](#5-cnn)
6. [NLP — Embeddings, Attention & Transformers](#6-nlp)
7. [PyTorch Lightning — Clean, Scalable Training](#7-lightning)
8. [Hugging Face Transformers — Fine-Tuning BERT/GPT2](#8-huggingface)
9. [Advanced Autograd & Custom Ops](#9-autograd-advanced)
10. [Distributed Training — DDP & FSDP](#10-distributed)
11. [Mixed Precision & torch.compile](#11-optimization)
12. [TorchScript & ONNX Export](#12-deployment)
13. [Capstone Projects](#13-capstone)


---
## 1. PyTorch Fundamentals & Autograd
### ⏱ Estimated Time: 3 hours


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import time
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
matplotlib.rcParams['figure.dpi'] = 120

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch     : {torch.__version__}')
print(f'Device      : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name()}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# ── Tensors ───────────────────────────────────────────────────────────────────
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2) * 0.5

print('a:'); print(a)
print('\na + b:');       print(a + b)
print('\na @ b:');       print(torch.mm(a, b))
print('\na.T:');         print(a.T)
print('\nshape  :', a.shape)
print('dtype  :', a.dtype)
print('device :', a.device)

# Moving to device
a_dev = a.to(DEVICE)
print(f'\nOn {DEVICE}: {a_dev.device}')

# View vs reshape vs contiguous
x = torch.randn(4, 6)
print('\nx.view(2, 12) :', x.view(2, 12).shape)
print('x.reshape(3,8):', x.reshape(3, 8).shape)
print('x.flatten()   :', x.flatten().shape)


In [ ]:
# ── Autograd — compute graph and backward pass ───────────────────────────────
x = torch.tensor(3.0, requires_grad=True)
y = x**3 + 2*x**2 - x + 1
y.backward()
print(f'y(3) = {y.item():.2f}')
print(f'dy/dx at x=3 = {x.grad.item():.2f}  (expected: {3*9 + 4*3 - 1})')

# Jacobian computation
x = torch.randn(3, requires_grad=True)
y = x ** 2
jac = torch.autograd.functional.jacobian(lambda x: x**2, x)
print('\nJacobian of x² (diagonal matrix):')
print(jac)  # diagonal = 2x

# Detaching from computation graph
with torch.no_grad():
    z = x * 2   # no gradient tracking — use for inference
print('\nz.requires_grad:', z.requires_grad)  # False


In [ ]:
# ── Common autograd patterns ─────────────────────────────────────────────────
# 1. Zero gradients before backward (critical in training loops)
optimizer_ex = optim.Adam([torch.tensor(1.0, requires_grad=True)], lr=0.01)
optimizer_ex.zero_grad()   # DON'T forget this!

# 2. Gradient accumulation
model_ex = nn.Linear(10, 1)
accum_steps = 4
for i in range(accum_steps):
    x_ex = torch.randn(8, 10)
    loss  = model_ex(x_ex).sum()
    (loss / accum_steps).backward()   # divide loss, accumulate gradients
# Apply once after accumulation
print('Grad accumulated over 4 steps:')
print(model_ex.weight.grad.shape)

# 3. Gradient clipping
max_norm = 1.0
norm_before = model_ex.weight.grad.norm().item()
torch.nn.utils.clip_grad_norm_(model_ex.parameters(), max_norm)
norm_after = model_ex.weight.grad.norm().item()
print(f'\nGrad norm: {norm_before:.4f} → {norm_after:.4f} (clipped at {max_norm})')


---
## 2. nn.Module — Building Blocks
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── Every PyTorch model is an nn.Module ──────────────────────────────────────
class MLP(nn.Module):
    """Fully-connected network with BatchNorm + Dropout."""
    def __init__(self, in_features, hidden_sizes, out_features, dropout=0.3):
        super().__init__()
        layers = []
        sizes  = [in_features] + hidden_sizes
        for i in range(len(sizes) - 1):
            layers += [
                nn.Linear(sizes[i], sizes[i+1]),
                nn.BatchNorm1d(sizes[i+1]),
                nn.ReLU(),
                nn.Dropout(dropout)
            ]
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

mlp = MLP(20, [128, 64, 32], 1)
print(mlp)
print(f'\nTotal params: {sum(p.numel() for p in mlp.parameters()):,}')
print(f'Trainable:    {sum(p.numel() for p in mlp.parameters() if p.requires_grad):,}')

# Test forward pass
x_test = torch.randn(8, 20)
out = mlp(x_test)
print(f'\nInput: {x_test.shape}  →  Output: {out.shape}')


In [ ]:
# ── Hooks — intercept activations during forward/backward ───────────────────
activation_store = {}

def save_activation(name):
    def hook(module, input, output):
        activation_store[name] = output.detach()
    return hook

# Register hooks on each Linear layer
hooks = []
for name, module in mlp.named_modules():
    if isinstance(module, nn.Linear):
        hooks.append(module.register_forward_hook(save_activation(name)))

# Forward pass
with torch.no_grad():
    _ = mlp(torch.randn(4, 20))

print('Captured activations:')
for name, act in activation_store.items():
    print(f'  {name:25s}: {act.shape}  mean={act.mean():.4f}')

# Always remove hooks when done
for h in hooks:
    h.remove()


---
## 3. Custom Training Loop — from Scratch
### ⏱ Estimated Time: 4 hours


In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_sk, y_sk = make_classification(n_samples=6000, n_features=20,
                                    n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_sk, y_sk, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr).astype('float32')
X_te = scaler.transform(X_te).astype('float32')
y_tr = y_tr.astype('float32')
y_te = y_te.astype('float32')

train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds   = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=0, pin_memory=True)
print(f'Train batches: {len(train_dl)}   Val batches: {len(val_dl)}')


In [ ]:
# ── Production-grade training loop ───────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion, scheduler=None, scaler_amp=None):
    model.train()
    total_loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)  # slightly faster than zero_grad()
        if scaler_amp is not None:  # AMP training
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16):
                preds = model(xb).squeeze()
                loss  = criterion(preds, yb)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(optimizer)
            scaler_amp.update()
        else:
            preds = model(xb).squeeze()
            loss  = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item() * len(xb)
        correct    += ((preds > 0.5) == yb.bool()).sum().item()
        n          += len(xb)
    return total_loss / n, correct / n

@torch.inference_mode()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        preds = model(xb).squeeze()
        loss  = criterion(preds, yb)
        total_loss += loss.item() * len(xb)
        correct    += ((preds > 0.5) == yb.bool()).sum().item()
        n          += len(xb)
    return total_loss / n, correct / n

print('Training utilities defined ✓')


In [ ]:
# ── Run training with cosine annealing + AMP ─────────────────────────────────
model     = MLP(20, [256, 128, 64], 1, dropout=0.3).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS    = 50
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
amp_scaler= torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None

history  = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_dl, optimizer, criterion, None, amp_scaler)
    va_loss, va_acc = eval_epoch(model, val_dl, criterion)
    scheduler.step()

    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)

    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(), 'val_acc': va_acc},
                    '/tmp/best_model.pt')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS}  '
              f'loss: {tr_loss:.4f} acc: {tr_acc:.4f} | '
              f'val_loss: {va_loss:.4f} val_acc: {va_acc:.4f}')

print(f'\nBest val_acc: {best_acc:.4f}')
# Load best checkpoint
ckpt = torch.load('/tmp/best_model.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]+1}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(history['train_loss'], label='Train', lw=2)
ax1.plot(history['val_loss'],   label='Val',   lw=2, ls='--')
ax1.set_title('Loss', fontweight='bold'); ax1.legend()
ax2.plot(history['train_acc'], label='Train', lw=2)
ax2.plot(history['val_acc'],   label='Val',   lw=2, ls='--')
ax2.set_title('Accuracy', fontweight='bold'); ax2.legend()
plt.tight_layout(); plt.show()


---
## 4. DataLoader & Custom Datasets
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── Custom Dataset class ─────────────────────────────────────────────────────
from PIL import Image
import os, pathlib

class ImageFolderDataset(Dataset):
    """
    Expects directory structure:
        root/
          class_a/img1.jpg
          class_b/img2.png
    """
    def __init__(self, root, transform=None):
        self.root      = pathlib.Path(root)
        self.transform = transform
        self.classes   = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class2idx = {c: i for i, c in enumerate(self.classes)}
        self.samples   = [
            (p, self.class2idx[p.parent.name])
            for cls in self.classes
            for p in (self.root / cls).glob('*')
            if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp')
        ]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label


# ── DataLoader with custom collate ────────────────────────────────────────────
def variable_length_collate(batch):
    """Collate function for sequences of variable length."""
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=0)
    labels  = torch.tensor(labels)
    return padded, labels, lengths

print('Custom dataset & collate utilities defined.')

# ── torchvision transforms pipeline ──────────────────────────────────────────
import torchvision.transforms as T

train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # ImageNet stats
])
val_transform = T.Compose([
    T.Resize(256), T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print('Train transform:', train_transform)


---
## 5. Computer Vision — CNNs & ResNet
### ⏱ Estimated Time: 6 hours

### 🎯 Project A: Custom ResNet-20 on CIFAR-10


In [ ]:
import torchvision
import torchvision.transforms as T

cifar_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
cifar_val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
cifar_train = torchvision.datasets.CIFAR10(
    root='/tmp/cifar', train=True,  transform=cifar_transform, download=True
)
cifar_test  = torchvision.datasets.CIFAR10(
    root='/tmp/cifar', train=False, transform=cifar_val_transform, download=True
)
train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True,
                           num_workers=2, pin_memory=True)
test_loader  = DataLoader(cifar_test,  batch_size=256, shuffle=False,
                           num_workers=2, pin_memory=True)
print(f'CIFAR-10: {len(cifar_train)} train, {len(cifar_test)} test')
CIFAR_CLASSES = ['airplane','automobile','bird','cat','deer',
                  'dog','frog','horse','ship','truck']


In [ ]:
# ── ResNet-20 for CIFAR ───────────────────────────────────────────────────────
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))


class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem    = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16), nn.ReLU()
        )
        self.layer1  = self._make_layer(16,  16,  n=3, stride=1)
        self.layer2  = self._make_layer(16,  32,  n=3, stride=2)
        self.layer3  = self._make_layer(32,  64,  n=3, stride=2)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.fc      = nn.Linear(64, num_classes)

    def _make_layer(self, in_ch, out_ch, n, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(n - 1):
            layers.append(BasicBlock(out_ch, out_ch))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

resnet20 = ResNet20().to(DEVICE)
print(f'ResNet-20 params: {sum(p.numel() for p in resnet20.parameters()):,}')
dummy = torch.randn(2, 3, 32, 32, device=DEVICE)
print('Output shape:', resnet20(dummy).shape)


In [ ]:
# ── Train ResNet-20 ───────────────────────────────────────────────────────────
EPOCHS_CNN = 100
opt_cnn    = optim.SGD(resnet20.parameters(), lr=0.1,
                        momentum=0.9, weight_decay=5e-4, nesterov=True)
sched_cnn  = optim.lr_scheduler.MultiStepLR(opt_cnn, milestones=[60, 80], gamma=0.1)
loss_cnn   = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.inference_mode()
def cifar_accuracy(model, loader):
    model.eval()
    correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        preds = resnet20(xb).argmax(1)
        correct += (preds == yb).sum().item()
        n       += len(yb)
    return correct / n

amp_scaler_cnn = torch.cuda.amp.GradScaler() if DEVICE.type == 'cuda' else None
cnn_history = {'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS_CNN):
    resnet20.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt_cnn.zero_grad(set_to_none=True)
        if amp_scaler_cnn:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                loss = loss_cnn(resnet20(xb), yb)
            amp_scaler_cnn.scale(loss).backward()
            amp_scaler_cnn.unscale_(opt_cnn)
            torch.nn.utils.clip_grad_norm_(resnet20.parameters(), 1.0)
            amp_scaler_cnn.step(opt_cnn)
            amp_scaler_cnn.update()
        else:
            loss = loss_cnn(resnet20(xb), yb)
            loss.backward()
            opt_cnn.step()
    sched_cnn.step()

    if (epoch + 1) % 20 == 0:
        tr_acc = cifar_accuracy(resnet20, train_loader)
        va_acc = cifar_accuracy(resnet20, test_loader)
        cnn_history['train_acc'].append(tr_acc)
        cnn_history['val_acc'].append(va_acc)
        print(f'Epoch {epoch+1:3d}  lr={opt_cnn.param_groups[0]["lr"]:.4f}  '
              f'train_acc={tr_acc:.4f}  val_acc={va_acc:.4f}')


### 🎯 Project B: Transfer Learning with pretrained ResNet50


In [ ]:
# ── Fine-tune ResNet50 (ImageNet weights) ─────────────────────────────────────
from torchvision.models import resnet50, ResNet50_Weights

def build_finetune_model(num_classes=10, freeze_up_to_layer='layer3'):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

    # Freeze everything
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze from freeze_up_to_layer onwards
    unfreeze = False
    for name, module in model.named_children():
        if name == freeze_up_to_layer:
            unfreeze = True
        if unfreeze:
            for p in module.parameters():
                p.requires_grad = True

    # Replace final FC
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, num_classes)
    )
    return model

tl_model = build_finetune_model().to(DEVICE)
trainable = sum(p.numel() for p in tl_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in tl_model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')

# Differential learning rates: lower LR for backbone, higher for head
backbone_params = [p for n, p in tl_model.named_parameters()
                   if p.requires_grad and 'fc' not in n]
head_params     = list(tl_model.fc.parameters())
tl_optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': 1e-5},
    {'params': head_params,     'lr': 1e-3}
], weight_decay=1e-4)
print('Differential LR optimizer set up ✓')


---
## 6. NLP — Embeddings, Attention & Transformers
### ⏱ Estimated Time: 8 hours

### 🎯 Project: Transformer from scratch — character-level language model


In [ ]:
# ── Scaled Dot-Product Attention ─────────────────────────────────────────────
def scaled_dot_product(q, k, v, mask=None):
    d_k     = q.size(-1)
    scores  = torch.matmul(q, k.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn    = F.softmax(scores, dim=-1)
    return torch.matmul(attn, v), attn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k    = d_model // n_heads
        self.n_heads= n_heads
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        Q = self.Wq(q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.Wk(k).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.Wv(v).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        ctx, attn = scaled_dot_product(Q, K, V, mask)
        ctx = ctx.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.dropout(self.Wo(ctx)), attn


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
        self.ln1   = nn.LayerNorm(d_model)
        self.ln2   = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        h, _  = self.attn(self.ln1(x), self.ln1(x), self.ln1(x), mask)
        x     = x + self.drop(h)
        x     = x + self.drop(self.ff(self.ln2(x)))
        return x

print('Transformer components defined ✓')
# Quick shape test
blk = TransformerBlock(d_model=128, n_heads=4, d_ff=512)
x_t = torch.randn(2, 16, 128)   # (batch, seq_len, d_model)
print('Output shape:', blk(x_t).shape)


In [ ]:
# ── GPT-style decoder language model ─────────────────────────────────────────
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=4,
                  d_ff=1024, max_len=256, dropout=0.1):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.blocks    = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)]
        )
        self.ln_f  = nn.LayerNorm(d_model)
        self.head  = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying: embed weights = unembedding weights
        self.tok_embed.weight = self.head.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        x    = self.tok_embed(idx) + self.pos_embed(pos)
        # Causal mask: prevent attending to future tokens
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)
        for block in self.blocks:
            x = block(x, mask)
        x      = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        return logits

    @torch.inference_mode()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=50):
        for _ in range(max_new_tokens):
            logits = self(idx[:, -256:])
            logits = logits[:, -1, :] / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx

print('GPT Language Model defined.')
gpt_demo = GPTLanguageModel(vocab_size=100, d_model=128, n_heads=4, n_layers=3)
print(f'Params: {sum(p.numel() for p in gpt_demo.parameters()):,}')


---
## 7. PyTorch Lightning — Clean, Scalable Training
### ⏱ Estimated Time: 5 hours

Lightning eliminates boilerplate while keeping full PyTorch flexibility.

```
Trainer handles: GPU/multi-GPU, logging, checkpointing, mixed precision, gradient clipping
You write     : LightningModule (forward + training_step + configure_optimizers)
```


In [ ]:
try:
    import lightning as L
    from lightning.pytorch.callbacks import (
        EarlyStopping, ModelCheckpoint, LearningRateMonitor, RichProgressBar
    )
    from lightning.pytorch.loggers import TensorBoardLogger
    print(f'Lightning version: {L.__version__}')
    LIGHTNING_AVAILABLE = True
except ImportError:
    print('Install: pip install lightning')
    LIGHTNING_AVAILABLE = False


In [ ]:
if LIGHTNING_AVAILABLE:

    class LitClassifier(L.LightningModule):
        def __init__(self, in_features, hidden, out_features, lr=1e-3):
            super().__init__()
            self.save_hyperparameters()   # logs all __init__ args automatically
            self.net = MLP(in_features, hidden, out_features)
            self.loss= nn.BCEWithLogitsLoss()

        def forward(self, x):
            return self.net(x)

        def _shared_step(self, batch, stage):
            x, y   = batch
            logits = self(x).squeeze()
            loss   = self.loss(logits, y)
            acc    = ((logits > 0) == y.bool()).float().mean()
            self.log(f'{stage}/loss', loss, prog_bar=True, on_epoch=True)
            self.log(f'{stage}/acc',  acc,  prog_bar=True, on_epoch=True)
            return loss

        def training_step(self, batch, batch_idx):
            return self._shared_step(batch, 'train')

        def validation_step(self, batch, batch_idx):
            self._shared_step(batch, 'val')

        def test_step(self, batch, batch_idx):
            self._shared_step(batch, 'test')

        def configure_optimizers(self):
            optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr,
                                     weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=self.trainer.max_epochs
            )
            return {'optimizer': optimizer,
                    'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}}

    lit_model = LitClassifier(20, [128, 64], 1, lr=1e-3)

    trainer = L.Trainer(
        max_epochs=50,
        accelerator='auto',
        precision='16-mixed' if DEVICE.type == 'cuda' else 32,
        gradient_clip_val=1.0,
        callbacks=[
            EarlyStopping(monitor='val/loss', patience=10, mode='min'),
            ModelCheckpoint(monitor='val/acc', mode='max',
                             dirpath='/tmp/lit_ckpts', filename='best-{epoch}-{val/acc:.3f}'),
            LearningRateMonitor(logging_interval='epoch'),
        ],
        logger=TensorBoardLogger('/tmp/lit_logs', name='classifier'),
        enable_progress_bar=True,
        log_every_n_steps=10
    )

    trainer.fit(lit_model, train_dataloaders=train_dl, val_dataloaders=val_dl)
    results = trainer.test(lit_model, dataloaders=val_dl)
    print('\nTest results:', results)
else:
    print('Skipping — Lightning not installed.')


---
## 8. Hugging Face Transformers — Fine-Tuning BERT & GPT-2
### ⏱ Estimated Time: 8 hours

### 🎯 Project: BERT Sequence Classification (SST-2 sentiment)


In [ ]:
try:
    from transformers import (
        BertTokenizerFast, BertForSequenceClassification,
        get_linear_schedule_with_warmup, DataCollatorWithPadding
    )
    from torch.optim import AdamW
    HF_AVAILABLE = True
    print('Hugging Face Transformers available ✓')
except ImportError:
    print('Install: pip install transformers datasets')
    HF_AVAILABLE = False


In [ ]:
if HF_AVAILABLE:

    BERT_MODEL = 'bert-base-uncased'
    tokenizer  = BertTokenizerFast.from_pretrained(BERT_MODEL)
    model_bert = BertForSequenceClassification.from_pretrained(
        BERT_MODEL, num_labels=2
    ).to(DEVICE)

    # Dummy dataset for demonstration
    texts  = [
        'This movie is absolutely fantastic!',
        'What a terrible waste of time.',
        'I loved every minute of this film.',
        'The worst acting I have ever seen.',
        'A masterpiece of modern cinema.',
        'Boring and predictable from start to finish.',
    ] * 100
    labels = [1, 0, 1, 0, 1, 0] * 100

    encodings = tokenizer(texts, truncation=True, padding=True,
                           max_length=128, return_tensors='pt')
    input_ids      = encodings['input_ids']
    attention_mask = encodings['attention_mask']
    labels_t       = torch.tensor(labels)

    bert_ds = TensorDataset(input_ids, attention_mask, labels_t)
    bert_dl = DataLoader(bert_ds, batch_size=16, shuffle=True)

    # AdamW with layer-wise LR decay
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_bert = AdamW([
        {'params': [p for n, p in model_bert.named_parameters()
                    if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in model_bert.named_parameters()
                    if any(nd in n for nd in no_decay)],     'weight_decay': 0.0},
    ], lr=2e-5)

    EPOCHS_BERT    = 3
    total_steps    = len(bert_dl) * EPOCHS_BERT
    warmup_steps   = total_steps // 10
    scheduler_bert = get_linear_schedule_with_warmup(
        optimizer_bert, warmup_steps, total_steps
    )

    model_bert.train()
    for epoch in range(EPOCHS_BERT):
        total_loss = 0
        for batch in bert_dl:
            ids, mask, lbls = [b.to(DEVICE) for b in batch]
            optimizer_bert.zero_grad(set_to_none=True)
            out  = model_bert(ids, attention_mask=mask, labels=lbls)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_bert.parameters(), 1.0)
            optimizer_bert.step()
            scheduler_bert.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1}/{EPOCHS_BERT}  avg_loss: {total_loss/len(bert_dl):.4f}')

    print('\nBERT fine-tuning complete ✓')
else:
    print('Skipping — Hugging Face not installed.')


In [ ]:
if HF_AVAILABLE:
    # Inference
    model_bert.eval()
    test_texts = [
        'Absolutely brilliant performance!',
        'I fell asleep halfway through.',
        'A stunning visual spectacle.',
    ]
    enc = tokenizer(test_texts, truncation=True, padding=True,
                     max_length=128, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        logits = model_bert(**enc).logits
        probs  = F.softmax(logits, dim=-1)
        preds  = logits.argmax(-1)

    label_map = {0: 'NEGATIVE', 1: 'POSITIVE'}
    for text, pred, prob in zip(test_texts, preds, probs):
        print(f'{label_map[pred.item()]:8s}  ({prob[pred].item():.3f})  "{text}"')


---
## 9. Advanced Autograd & Custom Ops
### ⏱ Estimated Time: 4 hours


In [ ]:
# ── Custom autograd Function ─────────────────────────────────────────────────
class StraightThroughEstimator(torch.autograd.Function):
    """Binary quantisation with straight-through gradient estimator."""
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.sign(x)   # binarise: {-1, +1}

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        # Only pass gradient where |x| <= 1 (clipped STE)
        grad_input = grad_output.clone()
        grad_input[x.abs() > 1] = 0
        return grad_input

ste = StraightThroughEstimator.apply
x   = torch.tensor([-1.5, -0.5, 0.0, 0.5, 1.5], requires_grad=True)
y   = ste(x)
y.sum().backward()
print('Input  :', x.data.numpy())
print('Forward:', y.data.numpy())  # {-1, -1, 1, 1, 1}
print('Grad   :', x.grad.numpy())  # STE: 0 where |x|>1, else 1


In [ ]:
# ── Vmap: batch over any function ────────────────────────────────────────────
from torch.func import vmap, grad, jacrev, functional_call

# Compute per-sample gradients efficiently
simple_model = nn.Linear(4, 2)
params = dict(simple_model.named_parameters())

def single_sample_loss(params, x, y):
    pred = functional_call(simple_model, params, x.unsqueeze(0)).squeeze()
    return F.mse_loss(pred, y)

per_sample_grad = vmap(grad(single_sample_loss), in_dims=(None, 0, 0))

x_batch = torch.randn(8, 4)
y_batch = torch.randn(8, 2)
grads_per_sample = per_sample_grad(params, x_batch, y_batch)
print('Per-sample gradient shapes:')
for k, v in grads_per_sample.items():
    print(f'  {k:15s}: {v.shape}')   # (8, out, in) and (8, out)


---
## 10. Distributed Training — DDP & FSDP
### ⏱ Estimated Time: 4 hours

| Method | Scale | Memory | Use Case |
|---|---|---|---|
| `DataParallel` | Single node | Replicated | Quick multi-GPU |
| `DistributedDataParallel` | Multi-node | Replicated | Standard multi-GPU |
| `FullyShardedDataParallel` | Multi-node | Sharded | Giant models (LLMs) |


In [ ]:
# ── DDP launcher script template ─────────────────────────────────────────────
DDP_SCRIPT = '''
# train_ddp.py  —  Run with: torchrun --nproc_per_node=4 train_ddp.py
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)

def cleanup():
    dist.destroy_process_group()

def train(rank, world_size, model_cls, dataset):
    setup(rank, world_size)
    device = torch.device(f"cuda:{rank}")

    model = model_cls().to(device)
    model = DDP(model, device_ids=[rank], find_unused_parameters=False)

    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader  = DataLoader(dataset, batch_size=128, sampler=sampler,
                          pin_memory=True, num_workers=4)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3 * world_size)

    for epoch in range(50):
        sampler.set_epoch(epoch)   # shuffle differently each epoch
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = torch.nn.functional.cross_entropy(model(xb), yb)
            loss.backward()   # DDP automatically averages gradients
            optimizer.step()

        # Only rank 0 saves checkpoints / logs
        if rank == 0 and (epoch + 1) % 10 == 0:
            torch.save(model.module.state_dict(), f"ckpt_epoch{epoch+1}.pt")

    cleanup()

if __name__ == "__main__":
    import torch.multiprocessing as mp
    world_size = torch.cuda.device_count()
    # mp.spawn(train, args=(world_size, MyModel, my_dataset), nprocs=world_size)
'''
print(DDP_SCRIPT)
with open('/tmp/train_ddp.py', 'w') as f:
    f.write(DDP_SCRIPT.strip())
print('DDP script saved to /tmp/train_ddp.py')


In [ ]:
# ── FSDP template for large models ───────────────────────────────────────────
FSDP_SNIPPET = '''
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import MixedPrecision, ShardingStrategy
from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy
import functools

# Auto-wrap each TransformerBlock as a separate FSDP unit
auto_wrap = functools.partial(
    transformer_auto_wrap_policy,
    transformer_layer_cls={TransformerBlock}
)

mp_policy = MixedPrecision(
    param_dtype=torch.float16,
    reduce_dtype=torch.float16,
    buffer_dtype=torch.float16,
)
model = FSDP(
    model,
    auto_wrap_policy=auto_wrap,
    mixed_precision=mp_policy,
    sharding_strategy=ShardingStrategy.FULL_SHARD,  # shard params + grads + optimizer
    device_id=torch.cuda.current_device(),
)
# Saving: gather all shards first
# from torch.distributed.fsdp import FullStateDictConfig, StateDictType
# with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT, FullStateDictConfig(...)):
#     state_dict = model.state_dict()
'''
print('FSDP template:')
print(FSDP_SNIPPET)


---
## 11. Mixed Precision & torch.compile
### ⏱ Estimated Time: 3 hours


In [ ]:
# ── torch.compile (PyTorch 2.0+) — 1.5–3× speedup ───────────────────────────
model_raw  = ResNet20().to(DEVICE)
model_comp = torch.compile(ResNet20().to(DEVICE), mode='reduce-overhead')

dummy_input = torch.randn(32, 3, 32, 32, device=DEVICE)

# Warm-up (compile happens on first call)
with torch.no_grad():
    model_comp(dummy_input)

N_BENCH = 100
# Raw
torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_BENCH):
        model_raw(dummy_input)
torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
t_raw = (time.perf_counter() - t0) / N_BENCH * 1000

# Compiled
torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(N_BENCH):
        model_comp(dummy_input)
torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
t_comp = (time.perf_counter() - t0) / N_BENCH * 1000

print(f'Raw model  : {t_raw:.3f} ms/batch')
print(f'Compiled   : {t_comp:.3f} ms/batch')
print(f'Speedup    : {t_raw/t_comp:.2f}×')


In [ ]:
# ── AMP inference with torch.autocast ────────────────────────────────────────
model_raw.eval()
with torch.inference_mode():
    # Full precision baseline
    t0 = time.perf_counter()
    for _ in range(N_BENCH):
        _ = model_raw(dummy_input)
    t_fp32 = (time.perf_counter() - t0) / N_BENCH * 1000

    # Mixed precision (float16 on CUDA, bfloat16 on CPU)
    device_type = DEVICE.type
    amp_dtype   = torch.float16 if device_type == 'cuda' else torch.bfloat16
    t0 = time.perf_counter()
    for _ in range(N_BENCH):
        with torch.autocast(device_type=device_type, dtype=amp_dtype):
            _ = model_raw(dummy_input)
    t_amp = (time.perf_counter() - t0) / N_BENCH * 1000

print(f'FP32 inference : {t_fp32:.3f} ms/batch')
print(f'AMP  inference : {t_amp:.3f} ms/batch')
print(f'Speedup        : {t_fp32/t_amp:.2f}×')


---
## 12. TorchScript & ONNX Export
### ⏱ Estimated Time: 3 hours

| Format | Size | Target Environment |
|---|---|---|
| TorchScript | ~same | C++ / mobile |
| ONNX | smaller | TensorRT, OpenVINO, CoreML, browser |
| TFLite | smallest | Mobile / embedded |


In [ ]:
# ── TorchScript — trace vs script ────────────────────────────────────────────
# Method 1: torch.jit.trace (works for static graphs)
example_input = torch.randn(1, 20)
model.eval()
traced = torch.jit.trace(model, example_input)
print('Traced model output:', traced(example_input).shape)

# Method 2: torch.jit.script (handles Python control flow)
class SimpleScriptable(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 5)

    def forward(self, x: torch.Tensor, use_relu: bool = True) -> torch.Tensor:
        out = self.fc(x)
        if use_relu:
            out = torch.relu(out)
        return out

scripted = torch.jit.script(SimpleScriptable())
print('Scripted model output:', scripted(torch.randn(2, 10)).shape)

# Save & load
traced.save('/tmp/traced_model.pt')
loaded_traced = torch.jit.load('/tmp/traced_model.pt')
diff = (loaded_traced(example_input) - traced(example_input)).abs().max()
print(f'\nMax diff after save/load: {diff.item():.2e}  (should be 0)')


In [ ]:
# ── ONNX export ───────────────────────────────────────────────────────────────
import torch.onnx

ONNX_PATH = '/tmp/model.onnx'
model.eval()
dummy_in = torch.randn(1, 20)  # representative input

torch.onnx.export(
    model,
    dummy_in,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)
import os
print(f'ONNX model: {os.path.getsize(ONNX_PATH) / 1024:.1f} KB')

# Verify with onnxruntime
try:
    import onnxruntime as ort
    sess  = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
    x_np  = dummy_in.numpy()
    onnx_out = sess.run(None, {'input': x_np})[0]
    pt_out   = model(dummy_in).detach().numpy()
    max_diff = np.abs(onnx_out - pt_out).max()
    print(f'ONNX vs PyTorch max diff: {max_diff:.2e}')
    print(f'ONNX providers: {ort.get_available_providers()}')
except ImportError:
    print('onnxruntime not installed. Run: pip install onnxruntime')


In [ ]:
# ── FastAPI serving template ──────────────────────────────────────────────────
PT_API_CODE = '''
# pytorch_api.py — uvicorn pytorch_api:app --host 0.0.0.0 --port 8000
import torch, numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app = FastAPI(title="PyTorch Model API", version="1.0")

# Load TorchScript model (thread-safe)
model = torch.jit.load("/tmp/traced_model.pt")
model.eval()

class InferenceRequest(BaseModel):
    inputs: List[List[float]]   # (batch, features)

class InferenceResponse(BaseModel):
    outputs: List[float]
    probabilities: List[float]

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/predict", response_model=InferenceResponse)
def predict(req: InferenceRequest):
    x    = torch.tensor(req.inputs, dtype=torch.float32)
    with torch.inference_mode():
        out = model(x).squeeze().tolist()
    prob = torch.sigmoid(torch.tensor(out)).tolist()
    return InferenceResponse(outputs=out if isinstance(out, list) else [out],
                              probabilities=prob if isinstance(prob, list) else [prob])
'''
print(PT_API_CODE)
with open('/tmp/pytorch_api.py', 'w') as f:
    f.write(PT_API_CODE.strip())
print('\nSaved to /tmp/pytorch_api.py')


---
## 13. Capstone Projects

### 🏆 Capstone A: ResNet-from-Scratch — CIFAR-10 ≥ 93%
```
ResNet-20 → CutMix + MixUp augmentation → Label smoothing
           → SAM optimizer → LR warmup + cosine decay
           → Model EMA → Test-Time Augmentation
           → Target: ≥ 93% top-1 accuracy
```

### 🏆 Capstone B: End-to-End NLP Pipeline
```
Raw text → Custom tokenizer → GPT-2 fine-tune
         → RLHF reward model (optional) → Beam search decoding
         → FastAPI chat endpoint → Streamlit demo
```

### 🏆 Capstone C: Production ML System
```
Train (DDP, 4×GPU) → TorchScript + ONNX
                   → FastAPI + Docker
                   → NGINX load balancer
                   → Prometheus/Grafana monitoring
                   → GitHub Actions CI/CD
                   → Rolling deployment on Kubernetes
```

---

## 📋 PyTorch Best Practices Cheat Sheet

```python
# ── Training ──────────────────────────────────────────────────────────────────
optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()
with torch.autocast(device_type='cuda', dtype=torch.float16):  # AMP
    loss = criterion(model(x), y)
scaler.scale(loss).backward()
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer); scaler.update()

# ── Inference ─────────────────────────────────────────────────────────────────
model.eval()
with torch.inference_mode():   # faster than no_grad()
    out = model(x)

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False  # set True for fixed input size

# ── Memory ────────────────────────────────────────────────────────────────────
torch.cuda.empty_cache()
torch.cuda.memory_summary(device)
```

---
*PyTorch Deep Learning Curriculum — Complete ✓*
